![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 37 -- Lab 1: Long Words in Pieces

German glues short words together into long ones: Fuß and Ball make Fußball, and Haus, Tür and Schlüssel make Haustürschlüssel. A spell checker, a search engine or a reading app for learners needs the pieces back. Your goal is a model that reads a long word letter by letter and marks where each piece ends, for words it has never seen. Your file is scored with the mean, over the words, of the F1 of their pieces.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** German long words, each with one label per letter. The setup cell leaves three files in `DATA_DIR`: `train.csv` (an `id`, the `word`, and its `labels`: one digit per letter, separated by spaces, 1 where a piece of the word ends and 0 elsewhere), `test.csv` (the same without the labels) and `sample_submission.csv` (the format of the file you hand in).

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="nlp-02", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/long-words-in-pieces")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The starter solution

Every olympiad task comes with a starter solution: a short notebook that reads the files and hands in a valid file. Its score is the **baseline**, and your points count up from it. This is the starter of this task. Read it before you begin; you may reuse any of its lines.

```python
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(0)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
chars = sorted(set("".join(train["word"])))
char_to_id = {c: i + 1 for i, c in enumerate(chars)}


def encode(word):
    return torch.tensor([char_to_id.get(c, 0) for c in word])


X = pad_sequence([encode(w) for w in train["word"]], batch_first=True)
Y = pad_sequence([torch.tensor([float(b) for b in s.split()]) for s in train["labels"]], batch_first=True)

# Every letter is decided alone: an embedding, then one score per letter.
model = nn.Sequential(nn.Embedding(len(chars) + 1, 16, padding_idx=0), nn.Linear(16, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
for start in range(0, len(X), 256):
    loss = loss_fn(model(X[start:start + 256]).squeeze(-1), Y[start:start + 256])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

labels = []
with torch.no_grad():
    for word in test["word"]:
        bits = (torch.sigmoid(model(encode(word)).squeeze(-1)) > 0.5).int().tolist()
        labels.append(" ".join(str(b) for b in bits))
submission = pd.DataFrame({"id": test["id"], "labels": labels})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at some words before you build anything.

## Task 1: Load the three files

Read `train.csv`, `test.csv` and `sample_submission.csv` from `DATA_DIR` with `pd.read_csv` into `train`, `test` and `sample`. Print the three shapes.

Syntax hint (the shape of the call, not the answer):

```python
table = pd.read_csv(os.path.join(DATA_DIR, "name.csv"))
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, test=test, sample=sample)


## Task 2: Look at the words

Store the first five rows of `train` in `first_rows` and the number of rows of `train` in `n_rows`. Then print how many letters the words have (the smallest, the mean and the largest count), how many pieces they have (the number of 1s in `labels`: how many words have each number), and five words of `train` with a `|` at every cut, like `Haus|tür`.

Syntax hint (the shape of the call, not the answer):

```python
counts = table["column"].str.count("x").value_counts()
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_rows, size=n_rows)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 7. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears.

The scorer of this task, in plain Python. Run the cell; you will use `score(true_labels, pred_labels)` all afternoon.

In [ ]:
def pieces(bits):
    """The pieces that a list of 0 and 1 cuts a word into, as (start, end) pairs: a 1 ends a piece."""
    found, start = set(), 0
    for i, b in enumerate(bits):
        if b == 1:
            found.add((start, i + 1))
            start = i + 1
    if start < len(bits):
        found.add((start, len(bits)))
    return found


def word_f1(true_bits, pred_bits):
    """The F1 of one word: a predicted piece is right only when a true piece has the same start and end."""
    if len(pred_bits) != len(true_bits):
        return 0.0
    true_pieces, pred_pieces = pieces(true_bits), pieces(pred_bits)
    right = len(true_pieces & pred_pieces)
    if right == 0:
        return 0.0
    precision, recall = right / len(pred_pieces), right / len(true_pieces)
    return 2 * precision * recall / (precision + recall)


def score(true_labels, pred_labels):
    """The score of a file: the mean of word_f1 over the words. Labels are texts like "0 0 1 0 1"."""
    return float(np.mean([word_f1([int(b) for b in t.split()], [int(b) for b in p.split()])
                          for t, p in zip(true_labels, pred_labels)]))


print(score(["0 0 1 0 1"], ["0 0 1 0 1"]), score(["0 0 1 0 1"], ["0 0 0 0 1"]))


## Task 3: What the scorer computes

The cell above is the scorer of this task. Which sentence says what it computes?

- **a** The share of letters whose label (0 or 1) is right, over all the letters of the file.
- **b** For every word, the F1 of its pieces, where a predicted piece counts only if a true piece starts and ends at the same letters; then the mean over the words.
- **c** For every word, the F1 of the positions of its 1s; then the mean over the words.

Store the letter in `formula`.

In [ ]:
# Your code here


## Task 4: Two answers without a model

What would two answers that need no model score? `no_cut` leaves every word whole: 0 on every letter but the last. `cut_in_half` cuts every word once, after its first `len(word) // 2` letters (and puts the 1 on the last letter too). Score both on the words of `train` with `score` and store the two numbers in the dictionary `trivial` under these two names.

Syntax hint (the shape of the call, not the answer):

```python
made = table["column"].map(my_function)      # my_function applied to every row
```

In [ ]:
# Your code here


## Task 5: What the metric rewards

A word has two true pieces. Your model cuts it at the right place, and also at one wrong place. What does that word score?

- **a** 1.0: the right cut is there.
- **b** 0.4: one of its three predicted pieces is right (precision 1/3) and one of the two true pieces is found (recall 1/2).
- **c** 0.8: two of its three 1s are right, and both true 1s are found.

Store the letter in `rewards`.

In [ ]:
# Your code here


## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, and `id_col`, the name of its id column. Read both from the tables you loaded; do not type the number.

In [ ]:
# Your code here


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
# Your code here


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- A first file

The simplest pipeline that ends in a file comes first: a validation set, batches, a small recurrent model, a short training loop, a score, a file. Aim to hand in your first file within the first hour; this morning's last section is your toolbox.

## Task 8: Hold out a validation set

Split `train` into `tr` (90%) and `va` (10%) with `train_test_split` and `random_state=0`. No word may be in both. Print the two shapes.

Syntax hint (the shape of the call, not the answer):

```python
part_a, part_b = train_test_split(table, test_size=0.3, random_state=0)
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=tr["id"], val_ids=va["id"])


## Task 9: Letters to ids

Make the dictionary `char_to_id` that gives every letter of the training words a whole number from 1 up; 0 stays free for padding. Store in `n_chars` the number of ids including the 0. Print it.

In [ ]:
# Your code here


## Task 10: A dataset of words

Write a `Dataset` class `Words(words, labels)` whose item `i` is two tensors: the ids of the letters of word `i`, and its labels as floats (one per letter). A letter the dictionary does not know gets id 0. Then write `collate(batch)`, which pads the words of a batch and their labels to the longest word of the batch.

In [ ]:
# Your code here


## Task 11: One batch

Make `train_loader`, a `DataLoader` over `Words` of `tr` with `collate_fn=collate`, a batch size of your choice and `shuffle=True`. Take its first batch into `xb` and `yb` and print their shapes and data types.

Syntax hint (the shape of the call, not the answer):

```python
first_x, first_y = next(iter(loader))
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(xb, yb)


## Task 12: A recurrent tagger

Write a model class with `__init__` and `forward` that turns a batch of letter ids of shape (batch, letters) into one raw score per letter, shape (batch, letters). Store one in `model` (on `device`) and print the shape of `model(xb.to(device))`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, xb.to(device))


## Task 13: The training loop

Train `model` on `train_loader` with a loss that ignores the padding places, for a small number of epochs first (check the time of one epoch). Append the loss of every step to the list `losses`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.loop(losses)


## Task 14: From scores to labels

Write `predict(model, words)`, which returns for every word the probability that a piece ends at each of its letters, and `to_labels(probs)`, which turns one word's probabilities into a labels text like `"0 0 0 1 0 0 1"` with one digit per letter. Print the labels your model gives `Haustür`.

In [ ]:
# Your code here


## Task 15: Score the validation set

Turn the words of `va` into labels texts in `val_pred` and store their `score` against `va["labels"]` in `val_score`, a number between 0 and 1.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="segment_f1")


## Task 16: Write the submission file

Predict the words of `test` and write `submission.csv` with the columns of `sample`: the `id` of every test word and its `labels`. Leave the index out. Print the shape and the first rows.

Syntax hint (the shape of the call, not the answer):

```python
table.to_csv("name.csv", index=False)
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 4 -- Debugging drill

A classmate wrote the model below for this task, and it stops with an error. Run the first cell: it starts the drill's clock. Then repair the classmate's cell until it runs and prints the shape of its output, one score per letter of each word, and run the check. The clock stops at the first check that passes. Read the error message before you change anything.

In [ ]:
ck.drill_start("bug-shp-1")


## Task 17: Repair the classmate's model

Change as little as you can: the words, the ids and the sizes stay as they are.

In [ ]:
drill_words = ["Haustür", "Sonnenschein", "Fußballspieler", "Apfelbaum"]
drill_ids = {c: i + 1 for i, c in enumerate(sorted(set("".join(drill_words))))}
drill_x = pad_sequence([torch.tensor([drill_ids[c] for c in w]) for w in drill_words], batch_first=True)


class ClassmateTagger(nn.Module):
    def __init__(self, n_chars, emb=16, hidden=48):
        super().__init__()
        self.emb = nn.Embedding(n_chars, emb, padding_idx=0)
        self.rnn = nn.GRU(emb, hidden, num_layers=2, batch_first=True)
        self.head = nn.Linear(emb, 1)
    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.head(out).squeeze(-1)


torch.manual_seed(0)
drill_out = ClassmateTagger(len(drill_ids) + 1)(drill_x)
print(drill_out.shape)


In [ ]:
ck.drill_check("bug-shp-1", drill_out)


---
# Part 5 -- Improve

You have a file on the board. Now make it better, one change at a time, and let the validation set decide which change stays.

## Task 18: Compare changes on the validation set

Try at least two changes to your pipeline. Train each on `tr`, score it on `va` with `score`, and print the scores side by side.

In [ ]:
# Your code here


## Task 19: Hand in your best file

Build your best pipeline, keep its trained model in `final_model`, and write `submission.csv` again. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 6 -- Challenge: the longest words

Your model learned from words of a few pieces. Time to test it on monsters.

## Task 20: Cut the longest words

Print each of these words cut by `final_model`, with a `|` at every cut: Donaudampfschifffahrtsgesellschaft, Kraftfahrzeughaftpflichtversicherung, Rindfleischetikettierungsüberwachungsaufgabenübertragungsgesetz, Schokoladenkuchenrezept, Fußballweltmeisterschaftsendspiel. Then add two long words of your own, glued from pieces you find in `train`. Which word fools your model, and where?

In [ ]:
# Your code here


---
## Task 21: Reflect

Which change raised your validation score most, and why do you think it helped? Look at five words of `va` your model cut wrongly: what do they have in common? Which long word fooled your model, and what does that tell you about a model trained on shorter words?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task**: the scorer read as code, two answers without a model, what one wrong cut costs, and the format of the file, all before any model.
- **Letters to tensors**: ids for letters, a `Dataset`, and a `collate_fn` that pads every batch.
- **A recurrent tagger**: an embedding, an LSTM and one score per letter, trained with a loss that skips the padding, in a loop written out.
- **Debugging**: a shape error read, located and repaired against the clock.
- **Improve with the validation set**: changes compared on `va`, then the best pipeline handed in.

---

Made By **Sattam Altwaim**